# Part 3 in code — world model, corrected objective, and proposition search

Part 3 is where the framework becomes operational:
encode the observer-side object, encode the proposition, compute the interaction, predict the next task-relevant state, decode the visible consequences, then update.

And yes, this is also where the objective sign has to stop fighting the optimizer.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

results_dir = project_root / "results"
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

In [2]:
loss_summary = pd.read_csv(results_dir / "loss_search_summary.csv")
slow_summary = pd.read_csv(results_dir / "slow_update_summary.csv")
shock_summary = pd.read_csv(results_dir / "slow_update_shock_summary.csv")

loss_summary

,form,equation,val_main_bce,val_probe_bce,weight_norm
0,plus_probe_minus_reg,L = main + λ_probe * probe - λ_reg * reg,0.127731,0.138925,7.091701
1,plus_plus_no_reg,L = main + λ_probe * probe,0.144923,0.153414,5.465180
2,plus_plus,L = main + λ_probe * probe + λ_reg * reg,0.163360,0.170601,4.637855
3,main_only,L = main + λ_reg * reg,0.178096,0.200123,4.033128
4,current_minus_minus,L = main - λ_probe * probe - λ_reg * reg,0.428301,13.475081,2.360341
5,minus_probe_plus_reg,L = main - λ_probe * probe + λ_reg * reg,0.446492,9.066950,1.653449


## Why the positive-sum loss is the keeper

There is one numerically strong anti-regularized variant in the search.
I did not keep it.

The project keeps the version that aligns with the paper’s own prose:
probe losses are real losses, regularization is ordinary weight control, and the whole thing is minimized by gradient descent.

In [3]:
loss_summary.loc[loss_summary["form"].isin(["plus_plus", "current_minus_minus", "plus_probe_minus_reg"])]

,form,equation,val_main_bce,val_probe_bce,weight_norm
0,plus_probe_minus_reg,L = main + λ_probe * probe - λ_reg * reg,0.127731,0.138925,7.091701
2,plus_plus,L = main + λ_probe * probe + λ_reg * reg,0.163360,0.170601,4.637855
4,current_minus_minus,L = main - λ_probe * probe - λ_reg * reg,0.428301,13.475081,2.360341


## Why the EMA slow update is the keeper

A durable embedding should move when durable evidence accumulates.
It should not get rewritten like a goldfish every time one event arrives.

In [4]:
slow_summary, shock_summary

(                    form  \
 0         additive_delta   
 1             ema_target   
 2       overwrite_target   
 3  literal_current_delta   
 4              no_update   
 
                                                               equation  \
 0            \hat T \leftarrow \hat T + \alpha (\hat T^{new} - \hat T)   
 1              \hat T \leftarrow (1-\alpha)\hat T + \alpha\hat T^{new}   
 2                                       \hat T \leftarrow \hat T^{new}   
 3  \hat T \leftarrow (1-\alpha)\hat T + \alpha (\hat T^{new} - \hat T)   
 4                                             \hat T \leftarrow \hat T   
 
        rmse  avg_step  cp_error  
 0  0.138977  0.060179  0.203929  
 1  0.138977  0.060179  0.203929  
 2  0.178985  0.473161  0.346196  
 3  0.244484  0.063179  0.426289  
 4  0.435279  0.000000  0.795062  ,
                     form  \
 0             ema_target   
 1         additive_delta   
 2  literal_current_delta   
 3       overwrite_target   
 
              

## Projection equivalence and proposition search

If two propositions share the same task projection, they are equivalent for that task.
Once that is true, proposition search is just scoring admissible candidates under the model.

In [5]:
from gids_observer_framework.state import OperationalState, best_proposition

transition_params = {
    "A_z": np.array([[1.0, -0.4], [0.2, 0.7]]),
    "B_x": np.array([[0.6, 0.0], [0.0, 0.5]]),
    "C_fast": np.zeros((2, 12)),
    "D_T": np.zeros((2, 2)),
    "E_c": np.zeros((2, 3)),
    "F_w": np.zeros((2, 2)),
}

state = OperationalState(
    T_hat=np.zeros(2),
    z_fast=np.array([0.3, 0.8]),
    context=np.zeros(3),
    world=np.zeros(2),
)

candidates = {
    "credibility": np.array([0.2, 0.1]),
    "urgency": np.array([1.2, -0.4]),
    "price": np.array([0.5, 0.9]),
}

best_name, scores = best_proposition(
    state=state,
    propositions=candidates,
    g_fast=np.zeros(12),
    transition_params=transition_params,
    readout_weights=np.array([1.2, -0.5]),
)

pd.DataFrame({"candidate": list(scores.keys()), "score": list(scores.values())}).sort_values("score", ascending=False), best_name

(     candidate     score
 1      urgency  0.628731
 2        price  0.483195
 0  credibility  0.456885,
 'urgency')

## Operational reading

The world model in this project is intentionally small and inspectable.

That is not a limitation.
It is the point.

Before you throw a giant sequence model at a domain, you should be able to say in plain English:
- what counts as slow state,
- what counts as fast state,
- what the proposition is,
- what the readout is,
- and which objective signs are actually coherent.